**Data** **Loading**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_california_housing

# Load dataset
data = fetch_california_housing()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['median_house_value'] = data.target

# Inspect the first few rows
df.head()

**Exploratory Data Analysis (EDA)**

In [ ]:
# Check for correlations
plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(), annot=True, cmap='YlGnBu')
plt.title("Correlation Heatmap")
plt.show()

# Visualize distribution of the target variable
sns.histplot(df['median_house_value'], kde=True)
plt.title("Distribution of House Prices")
plt.show()

**Data Preprocessing & Feature Engineering**

In [ ]:
# 1. Log Transformation to normalize skewed features
df['total_rooms'] = np.log1p(df['MedInc']) # Using MedInc as a proxy for wealth/size here
# Note: In the standard sklearn dataset, features are already pre-processed
# compared to the raw CSV, but let's add some engineered features:

df['bedroom_ratio'] = df['AveBedrms'] / df['AveRooms']
df['household_rooms'] = df['AveRooms'] / df['Population']

# 2. Handling Categorical Data (if using the raw CSV version with 'ocean_proximity')
# Since sklearn's version is purely numerical, we'll skip get_dummies
# unless you upload a custom CSV.

# 3. Train-Test Split
X = df.drop(['median_house_value'], axis=1)
y = df['median_house_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


**Model Training & Evaluation**

Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression

reg = LinearRegression()
reg.fit(X_train, y_train)

print(f"Linear Regression R^2 Score: {reg.score(X_test, y_test):.4f}")

Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

forest = RandomForestRegressor(n_estimators=100, random_state=42)
forest.fit(X_train, y_train)

print(f"Random Forest R^2 Score: {forest.score(X_test, y_test):.4f}")

**Hyperparameter Tuning (Optimization)**

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [50, 100],
    'max_features': [4, 8],
    'min_samples_split': [2, 4]
}

grid_search = GridSearchCV(RandomForestRegressor(), param_grid, cv=5, scoring='neg_mean_squared_error', return_train_score=True)
grid_search.fit(X_train, y_train)

best_forest = grid_search.best_estimator_
print(f"Optimized Score: {best_forest.score(X_test, y_test):.4f}")

**Model Evaluation & Error Analysis**

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Predict using the optimized model
y_pred = best_forest.predict(X_test)

# Calculate metrics
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)

print(f"Mean Absolute Error: ${mae * 100000:.2f}") # Prices are in units of $100,000
print(f"Root Mean Squared Error: ${rmse * 100000:.2f}")

# Visualization: Actual vs Predicted
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], '--r', linewidth=2)
plt.xlabel('Actual Prices')
plt.ylabel('Predicted Prices')
plt.title('Actual vs. Predicted House Prices')
plt.show()

**Feature Importance**

In [ ]:
importances = best_forest.feature_importances_
feature_names = X.columns
forest_importances = pd.Series(importances, index=feature_names).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=forest_importances, y=forest_importances.index)
plt.title("Feature Importance (Random Forest)")
plt.xlabel("Importance Score")
plt.show()

**Conclusion & Model Persistence**

In [ ]:
import joblib

# 1. Final Insights
print("--- PROJECT SUMMARY ---")
top_feature = forest_importances.index[0]
print(f"The most significant predictor for house prices is: {top_feature}")
print(f"Final Model Accuracy (R^2): {best_forest.score(X_test, y_test):.4f}")

# 2. Save the model for GitHub
joblib.dump(best_forest, 'california_housing_model.pkl')
print("Model saved as california_housing_model.pkl")